## Install, then use Runtime > Restart session

In [ ]:
!nvidia-smi
!pip uninstall -y pycolmap
!pip install -q pycolmap-cuda12==4.2.1

## Check and mount

In [ ]:
# using GPU
# import shutil, subprocess, time
# from pathlib import Path

# assert shutil.which("nvidia-smi"), "No GPU attached: switch to a GPU runtime"
# print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=True).stdout)

# import pycolmap
# print(pycolmap.__version__, "| has_cuda:", pycolmap.has_cuda)
# assert pycolmap.__version__ == "4.2.1" and pycolmap.has_cuda

# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
# using CPU
import shutil, time
from pathlib import Path
import pycolmap
print(pycolmap.__version__, "| has_cuda:", pycolmap.has_cuda)
assert pycolmap.__version__ == "4.2.1" and pycolmap.has_cuda

from google.colab import drive
drive.mount("/content/drive")

## Paths and helpers

In [ ]:
ROOT  = Path("/content/drive/MyDrive/Colab Notebooks/MSDS688/Case Study")
SCENE = "IMG_9175"
SRC       = ROOT / "data/colmap" / SCENE / "run_full_colab/undistorted"  # read only
DRIVE_OUT = ROOT / "data/mesh" / SCENE / "dense_input"
WS      = Path("/content/dense_ws") / SCENE
TEST_WS = Path("/content/dense_test") / SCENE

MAX_SIZE = 1000   # starting point: longest image side in pixels
CACHE_GB = 8      # starting point: default is 32

def make_workspace(dst):
    shutil.copytree(SRC, dst, dirs_exist_ok=True)
    for sub in ("depth_maps", "normal_maps", "consistency_graphs"):
        (dst / "stereo" / sub).mkdir(parents=True, exist_ok=True)
    n = len(list((dst / "images").glob("*.jpg")))
    assert n == 200, f"expected 200 images, found {n}"

def pm_options(geom):
    o = pycolmap.PatchMatchOptions()
    o.max_image_size = MAX_SIZE
    o.cache_size = CACHE_GB
    o.geom_consistency = geom
    return o

## Time three images first

In [ ]:
make_workspace(TEST_WS)
refs = ["frame_0000.jpg", "frame_0066.jpg", "frame_0133.jpg"]
(TEST_WS / "stereo/patch-match.cfg").write_text(
    "".join(f"{r}\n__auto__, 20\n" for r in refs))
t = time.time()
pycolmap.patch_match_stereo(TEST_WS, options=pm_options(geom=False))
per = (time.time() - t) / len(refs)
print(f"{per:.0f} s per image, one pass | rough full run: {per * 200 * 2 / 3600:.1f} h")

## Full patch-match

In [ ]:
make_workspace(WS)
pycolmap.patch_match_stereo(WS, options=pm_options(geom=True))

## Fuse and save to Drive

In [ ]:
fo = pycolmap.StereoFusionOptions()
fo.cache_size = CACHE_GB
fused = WS / "fused.ply"
pycolmap.stereo_fusion(fused, WS, options=fo, output_type="ply")
print(fused.read_bytes()[:600].split(b"end_header")[0].decode())   # count + fields
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(fused, DRIVE_OUT / "fused.ply")
print("saved", DRIVE_OUT / "fused.ply", f"{fused.stat().st_size / 1e6:.0f} MB")